# Base and tuned replies on a Kaggle T4 (generation only)

Generates every reply the one-command eval would score: the base model with the zero-shot prompt
and with the best ablation prompt, and the tuned adapter `hf:HUB_REPO`, over `EVAL_SET`. It runs
`eval.py --generate-only`, so it makes no judge calls and needs no API key. Judge the replies
locally:

```sh
uv run eval.py --judge-from <run_dir>/generations.jsonl
```

That writes the same `results.json`, `results.md` and `transcripts.jsonl` as the one-command
eval. `generation_pins.json` records the eval set hash, model revisions, chat-template hash,
decoding and library versions.

Before running:

- Settings: accelerator **GPU T4 x2** (the notebook uses one of the two), **Internet on**.
- Add-ons, Secrets (optional): `HF_TOKEN`, only if `HUB_REPO` is private.
- Set every line marked `# OWNER:` in the parameters cell. The notebook stops before doing
  anything while `REPO_URL` or `HUB_REPO` still holds the `OWNER` placeholder.

Afterwards, download the run directory under `/kaggle/working/eval` (both files) and run
`--judge-from` on it in the repo checkout at `REPO_COMMIT`.

In [ ]:
# OWNER: a clone URL this notebook can reach.
REPO_URL = "https://github.com/OWNER/small_model_finetune.git"
REPO_COMMIT = "dd36d336ea84e293b99727068507c420670111c6"
# OWNER: the tuned adapter on the Hub (append @<revision> to pin a commit).
HUB_REPO = "OWNER/groundcite-qwen3-1.7b-v1"
# OWNER: the eval set, relative to the repo at REPO_COMMIT.
EVAL_SET = "data/scenarios/heldout.jsonl"
OUT = "/kaggle/working/eval"

In [ ]:
# Fail before cloning or installing anything while a placeholder is still set.
for name in ("REPO_URL", "HUB_REPO"):
    if "OWNER" in globals()[name]:
        raise ValueError(f"{name} still holds the OWNER placeholder; set it in the parameters cell")

An optional `HF_TOKEN` secret, for a private adapter repo; nothing is printed.

In [ ]:
import os

from kaggle_secrets import UserSecretsClient

try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:  # not attached: public repos need no token
    pass

Check the GPU, then clone the repo at the pinned commit and install around Kaggle's torch.

In [ ]:
import os

import torch

# Kaggle's "GPU T4 x2" shows two GPUs; the model runs on one.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
if not torch.cuda.is_available():
    raise RuntimeError("no CUDA GPU: set Settings > Accelerator to GPU T4 x2")
print(torch.cuda.get_device_name(0), torch.__version__)

In [ ]:
import subprocess

REPO_DIR = "/tmp/repo"


def sh(cmd: str) -> None:
    subprocess.run(cmd, shell=True, check=True)


sh(f"git clone --quiet {REPO_URL} {REPO_DIR}")
sh(f"git -C {REPO_DIR} checkout --quiet {REPO_COMMIT}")
# An explicit pin list without torch, triton or nvidia-*: Kaggle's CUDA torch stays.
sh(f"pip install --quiet -r {REPO_DIR}/train/requirements-kaggle.txt")
sh(f"pip install --quiet --no-deps -e {REPO_DIR}")

Generate the base and tuned replies (fp16 on the T4, greedy, thinking off).

In [ ]:
generate = f"python eval.py --model hf:{HUB_REPO} --eval-set {EVAL_SET} --generate-only"
sh(f"cd {REPO_DIR} && {generate} --out {OUT}")